# Лабораторная работа №4

## Построение модели объектов: калибровка, стереозрение, облако точек

### Цель

Пройти путь от калибровки камеры до восстановления геометрии сцены: получить внутренние параметры с оценкой ошибки репроекции, построить карту диспаратности, восстановить облако точек и проверить глубину по объекту с известными размерами, а также оценить взаимное положение камер через эпиполярную геометрию.

Результатом работы является количественная оценка восстановленной глубины и вывод об ограничениях стереометода, а не одна визуально правдоподобная карта диспаратности.

[Методические указания блока](README.md) · [Общие МУ](../../../docs/guidelines-students.md) · [Рубрика оценивания](../teachers-assessment/README.md)

## 1. Что используется в работе

Библиотеки: `opencv-python`, `numpy`, `scikit-image`, `matplotlib`, `pandas`.

Данные. Ноутбук работает без интернета и использует три синтетических источника:

- **виды шахматной доски**, отрендеренные проецированием плоскости с известной матрицей внутренних параметров `K_true` — по ним выполняется калибровка, а результат сверяется с истиной;
- **стереопара с известной картой диспаратности**, построенная сдвигом пикселей по заданной карте — по ней считаются ошибка диспаратности, плотность карты и точность восстановленной глубины;
- **двухракурсные соответствия точек неплоской сцены** с известными $R$, $t$ и шумом с выбросами — по ним оценивается фундаментальная и существенная матрицы.

Ограничение синтетики названо прямо: рендер плоскости через гомографию **не моделирует дисторсию объектива**, поэтому коэффициенты дисторсии на синтетике будут близки к нулю, а ошибка репроекции — существенно ниже реальной. Синтетика проверяет корректность вашего конвейера; исследование дисторсии требует реальной съёмки. Как подставить собственную съёмку, описано в разделе 5.4; карточки данных (Middlebury, собственная стереосъёмка) — в [resources/datasets](../../resources/datasets/README.md).

Готовыми даны: генераторы всех трёх наборов, расчёт ошибки репроекции, метрики карты диспаратности, перевод диспаратности в облако точек, экспорт в PLY, визуализация эпиполярных линий, журнал экспериментов.

In [ ]:
# Зависимости (при необходимости раскомментируйте):
# %pip install opencv-python numpy scikit-image matplotlib pandas

import platform
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (регистрация 3D-проекции)
import skimage
from skimage import data

SEED = 42
rng = np.random.default_rng(SEED)
cv2.setRNGSeed(SEED)

OUTPUT_DIR = Path("outputs_lab4")
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)

print("python      :", platform.python_version())
print("opencv      :", cv2.__version__)
print("numpy       :", np.__version__)
print("scikit-image:", skimage.__version__)
print("pandas      :", pd.__version__)
print("SEED        :", SEED)

## 2. Краткая теоретическая справка

### 2.1. Модель камеры и калибровка

Проективная модель точечной камеры:

$$s\,\tilde{\mathbf{x}} = K\,[\,R \mid \mathbf{t}\,]\,\tilde{\mathbf{X}}, \qquad
K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix},$$

где $\tilde{\mathbf{X}}$ — однородные координаты точки сцены, $[\,R \mid \mathbf{t}\,]$ — внешние параметры вида, $K$ — внутренние параметры.

Радиальная и тангенциальная дисторсия описывается коэффициентами $k_1, k_2, p_1, p_2, k_3$:

$$x_d = x\,(1 + k_1 r^2 + k_2 r^4 + k_3 r^6) + 2p_1 x y + p_2 (r^2 + 2x^2), \qquad r^2 = x^2 + y^2 .$$

Качество калибровки характеризуется среднеквадратичной ошибкой репроекции:

$$e = \sqrt{\frac{1}{N}\sum_{i=1}^{N} \bigl\lVert \mathbf{x}_i - \hat{\mathbf{x}}_i(K, \mathbf{d}, R_i, \mathbf{t}_i) \bigr\rVert^2}.$$

Ошибка репроекции — обязательный отчётный показатель ([рубрика](../teachers-assessment/README.md#типичные-ошибки)). Её отсутствие делает калибровку непроверяемой. При этом малая ошибка сама по себе не гарантирует хорошей калибровки: она может означать недостаточное разнообразие ракурсов доски.

### 2.2. Эпиполярная геометрия

Для соответствующих точек $\mathbf{x}$ и $\mathbf{x}'$ двух видов выполняется эпиполярное ограничение:

$$\mathbf{x}'^{\mathsf{T}} F \mathbf{x} = 0 .$$

Матрица $F$ имеет ранг 2 и 7 степеней свободы. Прямая $\mathbf{l}' = F\mathbf{x}$ — эпиполярная линия во втором изображении, на которой обязан лежать образ точки $\mathbf{x}$: поиск соответствий сводится к одномерному.

Существенная матрица связана с фундаментальной через внутренние параметры:

$$E = K'^{\mathsf{T}} F K, \qquad E = [\mathbf{t}]_\times R,$$

где $[\mathbf{t}]_\times$ — кососимметричная матрица векторного произведения. Разница принципиальная: $F$ работает в пикселях и не требует калибровки, $E$ работает в нормированных координатах и позволяет извлечь $R$ и направление $\mathbf{t}$ (масштаб перемещения из двух видов не определяется).

Вырождение: если все точки сцены лежат в одной плоскости, оценка $F$ по соответствиям вырождена — семейство решений бесконечно. Именно поэтому набор для раздела 8 строится по неплоскому облаку точек.

### 2.3. Стереозрение и глубина

После ректификации соответствующие точки лежат на одной строке, и смещение вдоль строки — диспаратность $d = x_l - x_r$. Для параллельных камер с базой $B$ и фокусным расстоянием $f$ (в пикселях):

$$Z = \frac{f B}{d}, \qquad X = \frac{(u - c_x)\,Z}{f}, \qquad Y = \frac{(v - c_y)\,Z}{f}.$$

Отсюда следуют ограничения метода. Погрешность глубины растёт квадратично с расстоянием:

$$\left| \frac{\partial Z}{\partial d} \right| = \frac{fB}{d^2} = \frac{Z^2}{fB},$$

то есть ошибка в один пиксель диспаратности на дальнем плане стоит во много раз дороже, чем на ближнем. Однородные и повторяющиеся текстуры дают неоднозначное сопоставление, а области, видимые только одной камерой (полутени у границ объектов), не имеют соответствия в принципе.

### 2.4. Параметры блочного сопоставления

Размер блока задаёт компромисс: маленький блок сохраняет тонкие детали и границы, но чувствителен к шуму и даёт разреженную зашумлённую карту; большой блок сглаживает и «размазывает» границы объектов. Диапазон `numDisparities` ограничивает минимальную восстанавливаемую дальность: объекты ближе, чем $Z = fB/d_{\max}$, не будут найдены.

### 2.5. Метрики

Для карты диспаратности при известном эталоне:

$$\mathrm{MAE} = \frac{1}{|V|}\sum_{p \in V} \lvert d_p - d_p^{gt} \rvert, \qquad
\mathrm{Bad}_\theta = \frac{1}{|V|}\bigl|\{ p \in V : \lvert d_p - d_p^{gt}\rvert > \theta \}\bigr|,$$

где $V$ — множество пикселей с найденной диспаратностью. Отдельно приводится плотность $|V| / |\Omega|$: карта с малой ошибкой на 20 % пикселей и карта с той же ошибкой на 90 % — разные результаты.

## 3. Задачи

Формулировка из [методических указаний блока](README.md#лр4-построение-модели-объектов):

1. Откалибруйте камеру по шахматной доске (внутренние параметры, дисторсия).
2. Для стереопары выполните ректификацию, постройте карту диспаратности (StereoBM/StereoSGBM) и восстановите облако точек.
3. Исследуйте влияние параметров сопоставления на плотность и качество карты глубины.
4. По серии снимков объекта с разных ракурсов оцените взаимное положение камер через эпиполярную геометрию (фундаментальная/существенная матрица).

**Результат:** визуализация облака точек; количественная оценка глубины для объектов с известными размерами; вывод об ограничениях стереометода.

Проверяемые элементы ([рубрика](../teachers-assessment/README.md)): калибровка выполнена, ошибка репроекции указана; карта диспаратности построена и параметры исследованы серией; глубина проверена на объекте с известными размерами.

## 4. Журнал экспериментов

Каждая конфигурация сопоставления и каждый набор видов калибровки записываются в журнал. Без этого невозможно построить требуемую серию «параметр → плотность и качество карты».

In [ ]:
RUNS = []


def log_run(**fields):
    """Добавить запись в журнал экспериментов (конфигурация + метрики + время)."""
    record = {"run_id": len(RUNS), **fields}
    RUNS.append(record)
    return record


def runs_table(columns=None, sort_by=None):
    if not RUNS:
        return pd.DataFrame()
    frame = pd.DataFrame(RUNS)
    if columns:
        frame = frame[[c for c in columns if c in frame.columns]]
    if sort_by:
        frame = frame.sort_values(sort_by)
    return frame.reset_index(drop=True)


def save_runs(path=OUTPUT_DIR / "runs.csv"):
    runs_table().to_csv(path, index=False)
    return path

## 5. Калибровка камеры

### 5.1. Синтетические виды доски

Доска задаётся плоскостью $Z = 0$ в системе координат объекта. Для вида с параметрами $(R, \mathbf{t})$ проекция плоскости — гомография

$$H = K\,[\,\mathbf{r}_1\ \mathbf{r}_2\ \mathbf{t}\,]\,A^{-1},$$

где $A$ переводит координаты доски (в миллиметрах) в пиксели шаблона. Изображение вида получается переносом шаблона этой гомографией, поэтому истинные $K$, $R$, $\mathbf{t}$ известны точно.

In [ ]:
INNER_CORNERS = (9, 6)     # число внутренних углов: (по горизонтали, по вертикали)
SQUARE_MM = 25.0           # размер клетки, мм
IMAGE_SIZE = (640, 480)    # (ширина, высота)

K_TRUE = np.array([[800.0, 0.0, 320.0],
                   [0.0, 800.0, 240.0],
                   [0.0, 0.0, 1.0]])


def make_chessboard_template(inner=INNER_CORNERS, square_px=60, margin=60):
    """Изображение-шаблон шахматной доски и матрица перевода координат.

    Выход:
        template — (H, W) uint8, доска на белом поле;
        A        — (3, 3): координаты доски в мм -> пиксели шаблона.
    """
    cols, rows = inner[0] + 1, inner[1] + 1
    board = np.zeros((rows * square_px, cols * square_px), dtype=np.uint8)
    for i in range(rows):
        for j in range(cols):
            if (i + j) % 2 == 0:
                board[i * square_px:(i + 1) * square_px,
                      j * square_px:(j + 1) * square_px] = 255
    template = np.full((board.shape[0] + 2 * margin, board.shape[1] + 2 * margin),
                       255, dtype=np.uint8)
    template[margin:margin + board.shape[0], margin:margin + board.shape[1]] = board
    scale = square_px / SQUARE_MM
    A = np.array([[scale, 0.0, margin + square_px],
                  [0.0, scale, margin + square_px],
                  [0.0, 0.0, 1.0]])
    return template, A


def object_points(inner=INNER_CORNERS, square_mm=SQUARE_MM):
    """Координаты внутренних углов доски в системе объекта, мм. Выход: (N, 3) float32."""
    grid = np.zeros((inner[0] * inner[1], 3), np.float32)
    grid[:, :2] = np.mgrid[0:inner[0], 0:inner[1]].T.reshape(-1, 2) * square_mm
    return grid


def render_chessboard_views(n_views=15, K=K_TRUE, image_size=IMAGE_SIZE, seed=SEED):
    """Отрендерить виды доски с известными внутренними и внешними параметрами.

    Выход: (images list of (H, W) uint8, poses list of (rvec, tvec)).

    Дисторсия не моделируется: рендер строго проективный.
    """
    template, A = make_chessboard_template()
    generator = np.random.default_rng(seed)
    A_inv = np.linalg.inv(A)
    images, poses = [], []
    board_w = INNER_CORNERS[0] * SQUARE_MM
    board_h = INNER_CORNERS[1] * SQUARE_MM
    for _ in range(n_views):
        angles = generator.uniform(-0.45, 0.45, 3)
        rvec = angles.reshape(3, 1)
        R, _ = cv2.Rodrigues(rvec)
        distance = generator.uniform(600.0, 950.0)
        tvec = np.array([[-board_w / 2 + generator.uniform(-60, 60)],
                         [-board_h / 2 + generator.uniform(-45, 45)],
                         [distance]])
        P = K @ np.hstack([R[:, :2], tvec])
        H = P @ A_inv
        view = cv2.warpPerspective(template, H, image_size, flags=cv2.INTER_LINEAR,
                                   borderMode=cv2.BORDER_CONSTANT, borderValue=255)
        images.append(view)
        poses.append((rvec, tvec))
    return images, poses


views, true_poses = render_chessboard_views()
objp = object_points()

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, view in zip(axes.ravel(), views):
    ax.imshow(view, cmap="gray", vmin=0, vmax=255)
    ax.axis("off")
fig.suptitle("Синтетические виды шахматной доски")
plt.tight_layout()
plt.show()

In [ ]:
# Рельсы: поиск углов на одном виде с субпиксельным уточнением.
SUBPIX_CRITERIA = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

found, corners = cv2.findChessboardCorners(views[0], INNER_CORNERS, None)
print("Углы найдены:", found)

if found:
    refined = cv2.cornerSubPix(views[0], corners, (11, 11), (-1, -1), SUBPIX_CRITERIA)
    canvas = cv2.cvtColor(views[0], cv2.COLOR_GRAY2BGR)
    cv2.drawChessboardCorners(canvas, INNER_CORNERS, refined, found)
    plt.figure(figsize=(7, 5))
    plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    plt.title("Найденные внутренние углы (вид 0)")
    plt.axis("off")
    plt.show()

# Порядок углов, возвращаемый findChessboardCorners, согласован с порядком
# object_points(): нарушение этого соответствия — частая причина «странной» K.

In [ ]:
def reprojection_error(object_points_list, image_points_list, rvecs, tvecs,
                       camera_matrix, dist_coeffs):
    """Среднеквадратичная ошибка репроекции, пиксели.

    Выход: (total_rms, per_view list) — общая ошибка и ошибка по каждому виду.
    """
    per_view, squares, count = [], 0.0, 0
    for objp_i, imgp_i, rvec, tvec in zip(object_points_list, image_points_list, rvecs, tvecs):
        projected, _ = cv2.projectPoints(objp_i, rvec, tvec, camera_matrix, dist_coeffs)
        diff = projected.reshape(-1, 2) - imgp_i.reshape(-1, 2)
        view_squares = float(np.sum(diff ** 2))
        per_view.append(float(np.sqrt(view_squares / len(diff))))
        squares += view_squares
        count += len(diff)
    return float(np.sqrt(squares / count)), per_view


# TODO (задание 3.1): выполните калибровку.
#
# План:
#   1) соберите object_points_list и image_points_list по всем видам, где углы
#      найдены (виды с ненайденными углами не молча пропускайте, а посчитайте);
#   2) вызовите cv2.calibrateCamera(...) и получите K, dist, rvecs, tvecs;
#   3) выведите RMS из calibrateCamera и сравните её с reprojection_error(...);
#   4) сравните K с K_TRUE поэлементно: относительная ошибка f_x, f_y, c_x, c_y;
#   5) исследуйте влияние числа видов: 3, 5, 10, 15 — как меняются ошибка
#      репроекции и отклонение от K_TRUE;
#   6) залогируйте каждую конфигурацию через log_run(stage="calibration", ...).
#
# Ожидаемое на синтетике: dist близка к нулю, ошибка репроекции — доли пикселя.
# Если это не так, ищите ошибку в порядке точек или в типах данных (float32).

runs_table()

### 5.4. Подстановка реальной съёмки

Синтетика проверяет конвейер, но не заменяет реальную калибровку: в ней нет дисторсии, шума сенсора, расфокуса и неточностей печати доски.

Порядок работы с собственными данными (карточка «Собственная видеосъёмка / стереосъёмка» в [resources/datasets](../../resources/datasets/README.md)):

1. Распечатайте шахматную доску, наклейте на жёсткое основание, измерьте реальный размер клетки и подставьте его в `SQUARE_MM`.
2. Снимите 15–25 кадров доски: разные наклоны (в том числе сильные), разные расстояния, доска в разных частях кадра, обязательно с заходом на углы кадра — именно там проявляется дисторсия.
3. Загрузите кадры через `cv2.imread(..., cv2.IMREAD_GRAYSCALE)` и подставьте вместо `views`; остальной код не меняется.
4. Не смешивайте кадры, снятые с разными настройками камеры (зум, разрешение, автофокус): это разные камеры с точки зрения модели.
5. Для стереопары дополнительно снимите доску **одновременно** обеими камерами и используйте `cv2.stereoCalibrate` и `cv2.stereoRectify`; ректифицирующие отображения применяются через `cv2.initUndistortRectifyMap` и `cv2.remap`.

Типичная ошибка ([рубрика](../teachers-assessment/README.md#типичные-ошибки)): не указана ошибка репроекции калибровки. Указывайте и общую RMS, и распределение по видам — выброс по одному виду обычно означает неверно найденные углы.

## 6. Стереопара и карта диспаратности

Синтетическая стереопара строится переносом пикселей левого изображения по заданной карте диспаратности. Ближний объект рисуется последним, поэтому в правом изображении корректно возникают зоны перекрытия — области, видимые только левой камерой. Ни один алгоритм сопоставления не может найти в них соответствие: это ограничение сцены, а не метода.

Известны: фокусное расстояние в пикселях, база, эталонная диспаратность, а значит и точная глубина фона и объекта, и реальные размеры объекта.

In [ ]:
def make_synthetic_stereo(size=(288, 384), focal_px=700.0, baseline_mm=120.0,
                          bg_disparity=10, object_disparity=28, noise_sigma=3.0,
                          seed=SEED):
    """Синтетическая стереопара с известной картой диспаратности.

    Выход:
        left, right — (H, W, 3) uint8 BGR;
        disp_gt     — (H, W) float32, эталонная диспаратность в пикселях;
        meta        — словарь с параметрами камеры и истинными величинами объекта.
    """
    h, w = size
    generator = np.random.default_rng(seed)
    texture = cv2.cvtColor(data.astronaut(), cv2.COLOR_RGB2BGR)
    left = cv2.resize(texture, (w, h), interpolation=cv2.INTER_AREA).astype(np.float32)
    left = np.clip(left + generator.normal(0.0, 8.0, left.shape), 0, 255).astype(np.uint8)

    disp_gt = np.full((h, w), float(bg_disparity), dtype=np.float32)
    box = (int(0.30 * w), int(0.28 * h), int(0.34 * w), int(0.40 * h))
    bx, by, bw, bh = box
    disp_gt[by:by + bh, bx:bx + bw] = float(object_disparity)

    right = np.zeros_like(left)
    filled = np.zeros((h, w), dtype=bool)
    for value in sorted(np.unique(disp_gt).astype(int)):   # ближние объекты рисуются последними
        rows, cols = np.nonzero(disp_gt.astype(int) == value)
        target = cols - value
        ok = target >= 0
        right[rows[ok], target[ok]] = left[rows[ok], cols[ok]]
        filled[rows[ok], target[ok]] = True
    background = np.roll(left, -int(bg_disparity), axis=1)
    right[~filled] = background[~filled]
    right = np.clip(right.astype(np.float32) + generator.normal(0.0, noise_sigma, right.shape),
                    0, 255).astype(np.uint8)

    meta = {
        "focal_px": focal_px, "baseline_mm": baseline_mm,
        "cx": w / 2.0, "cy": h / 2.0, "image_size": (w, h),
        "object_box": box,
        "bg_disparity": bg_disparity, "object_disparity": object_disparity,
        "bg_depth_mm": focal_px * baseline_mm / bg_disparity,
        "object_depth_mm": focal_px * baseline_mm / object_disparity,
        "object_width_mm": bw * (focal_px * baseline_mm / object_disparity) / focal_px,
        "object_height_mm": bh * (focal_px * baseline_mm / object_disparity) / focal_px,
        "occlusion_share": float((~filled).mean()),
    }
    return left, right, disp_gt, meta


left, right, disp_gt, stereo_meta = make_synthetic_stereo()
for key, value in stereo_meta.items():
    print(f"{key:18s}: {value}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(cv2.cvtColor(left, cv2.COLOR_BGR2RGB))
axes[0].set_title("левое изображение")
axes[1].imshow(cv2.cvtColor(right, cv2.COLOR_BGR2RGB))
axes[1].set_title("правое изображение")
image = axes[2].imshow(disp_gt, cmap="magma")
axes[2].set_title("эталонная диспаратность")
plt.colorbar(image, ax=axes[2], fraction=0.046)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def disparity_metrics(disp_pred, disp_true, bad_threshold=2.0, ignore_border=16):
    """Метрики карты диспаратности.

    Вход:
        disp_pred — (H, W) float32, значения <= 0 считаются ненайденными;
        disp_true — (H, W) float32 эталон;
        bad_threshold — порог грубой ошибки, пиксели;
        ignore_border — сколько столбцов слева исключить (зона, где сопоставление
                        невозможно из-за диапазона поиска).
    Выход: словарь density, mae, rmse, bad_share, mean_depth_error_mm (для валидных).
    """
    valid = np.isfinite(disp_pred) & (disp_pred > 0)
    valid[:, :ignore_border] = False
    total = valid.size - valid.shape[0] * ignore_border
    if valid.sum() == 0:
        return {"density": 0.0, "mae": float("nan"), "rmse": float("nan"),
                "bad_share": float("nan")}
    error = np.abs(disp_pred[valid] - disp_true[valid])
    return {
        "density": round(float(valid.sum() / total), 4),
        "mae": round(float(error.mean()), 4),
        "rmse": round(float(np.sqrt((error ** 2).mean())), 4),
        "bad_share": round(float((error > bad_threshold).mean()), 4),
    }


def show_disparity(disp, title="", vmax=None):
    """Показать карту диспаратности; ненайденные значения выделены отдельно."""
    display_map = np.where(disp > 0, disp, np.nan)
    plt.figure(figsize=(6.5, 4.5))
    image = plt.imshow(display_map, cmap="magma", vmin=0,
                       vmax=vmax if vmax else np.nanmax(display_map))
    plt.colorbar(image, fraction=0.046)
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
# Рельсы: одна конфигурация StereoSGBM.
#
# Важно: compute() возвращает int16 с фиксированной точкой (значение умножено
# на 16), а ненайденные пиксели помечены отрицательными значениями. Без деления
# на 16 все метрики и все глубины окажутся неверными в 16 раз.
BLOCK = 7
sgbm = cv2.StereoSGBM_create(
    minDisparity=0,
    numDisparities=48,          # кратно 16
    blockSize=BLOCK,
    P1=8 * 3 * BLOCK ** 2,
    P2=32 * 3 * BLOCK ** 2,
    disp12MaxDiff=1,
    uniquenessRatio=10,
    speckleWindowSize=100,
    speckleRange=2,
)

start = time.perf_counter()
disp_sgbm = sgbm.compute(left, right).astype(np.float32) / 16.0
elapsed = time.perf_counter() - start

metrics = disparity_metrics(disp_sgbm, disp_gt)
log_run(stage="disparity", matcher="sgbm",
        params="numDisparities=48, blockSize=7, uniquenessRatio=10",
        seconds=round(elapsed, 4), seed=SEED, **metrics)

print("SGBM (baseline):", metrics)
show_disparity(disp_sgbm, "StereoSGBM, опорная конфигурация", vmax=float(disp_gt.max()))
runs_table()

In [ ]:
# TODO (задание 3.3): серия по параметрам сопоставления.
#
# Меняйте по одному фактору при фиксированных остальных:
#   1) blockSize in [5, 7, 11, 15, 21] (нечётный);
#   2) numDisparities in [16, 32, 48, 96] (кратно 16);
#   3) uniquenessRatio in [0, 5, 10, 20];
#   4) speckleWindowSize / speckleRange;
#   5) отдельно постройте карту StereoBM (cv2.StereoBM_create) на серых
#      изображениях и сравните с SGBM при сопоставимых параметрах.
#
# Для каждой конфигурации логируйте density, mae, rmse, bad_share и время.
# Ответьте измерением на вопрос защиты: как изменится карта при уменьшении
# блока сопоставления — что происходит с плотностью, с ошибкой и с границами
# объекта по отдельности.
#
# for block in [5, 7, 11, 15, 21]:
#     matcher = cv2.StereoSGBM_create(minDisparity=0, numDisparities=48, blockSize=block,
#                                     P1=8 * 3 * block ** 2, P2=32 * 3 * block ** 2, ...)
#     ...
#     log_run(stage="disparity", matcher="sgbm", params=f"blockSize={block}", ...)

pass

## 7. Облако точек и проверка глубины

Перевод диспаратности в трёхмерные координаты выполняется по формулам раздела 2.3. Эквивалентный путь в OpenCV — `cv2.reprojectImageTo3D` с матрицей $Q$ из `cv2.stereoRectify`; для синтетической пары с параллельными камерами обе записи совпадают.

Проверка глубины обязательна: сравните восстановленную глубину и восстановленный размер объекта с истинными значениями из `stereo_meta`. Для реальной съёмки роль эталона играет объект с измеренными линейкой размерами.

In [ ]:
def disparity_to_points(disp, meta, image_bgr=None, min_disparity=1.0):
    """Перевод карты диспаратности в облако точек.

    Вход:  disp (H, W) float32; meta — словарь из make_synthetic_stereo;
           image_bgr — изображение для раскраски точек или None.
    Выход: (points (N, 3) float32 в мм, colors (N, 3) uint8 RGB или None).
    """
    f = meta["focal_px"]
    baseline = meta["baseline_mm"]
    cx, cy = meta["cx"], meta["cy"]
    rows, cols = np.nonzero(np.isfinite(disp) & (disp > min_disparity))
    d = disp[rows, cols].astype(np.float64)
    Z = f * baseline / d
    X = (cols - cx) * Z / f
    Y = (rows - cy) * Z / f
    points = np.stack([X, Y, Z], axis=1).astype(np.float32)
    colors = None if image_bgr is None else image_bgr[rows, cols][:, ::-1].copy()
    return points, colors


def save_ply(points, colors, path):
    """Сохранить облако точек в ASCII PLY (открывается в MeshLab, CloudCompare)."""
    path = Path(path)
    with open(path, "w", encoding="ascii") as handle:
        handle.write("ply\nformat ascii 1.0\n")
        handle.write(f"element vertex {len(points)}\n")
        handle.write("property float x\nproperty float y\nproperty float z\n")
        if colors is not None:
            handle.write("property uchar red\nproperty uchar green\nproperty uchar blue\n")
        handle.write("end_header\n")
        for index in range(len(points)):
            x, y, z = points[index]
            if colors is None:
                handle.write(f"{x:.3f} {y:.3f} {z:.3f}\n")
            else:
                r, g, b = colors[index]
                handle.write(f"{x:.3f} {y:.3f} {z:.3f} {int(r)} {int(g)} {int(b)}\n")
    return path


def show_point_cloud(points, colors=None, max_points=8000, title="", elev=-70, azim=-90):
    """Быстрая визуализация облака точек (прореживание для читаемости)."""
    step = max(1, len(points) // max_points)
    sample = points[::step]
    sample_colors = None if colors is None else colors[::step] / 255.0
    figure = plt.figure(figsize=(7, 6))
    ax = figure.add_subplot(111, projection="3d")
    ax.scatter(sample[:, 0], sample[:, 1], sample[:, 2], s=1, c=sample_colors)
    ax.set_xlabel("X, мм")
    ax.set_ylabel("Y, мм")
    ax.set_zlabel("Z, мм")
    ax.view_init(elev=elev, azim=azim)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


# Рельсы: облако по эталонной диспаратности — верхняя граница качества.
points_gt, colors_gt = disparity_to_points(disp_gt, stereo_meta, left)
print("Точек в эталонном облаке:", len(points_gt),
      "| диапазон глубин, мм:", round(float(points_gt[:, 2].min()), 1),
      "-", round(float(points_gt[:, 2].max()), 1))
show_point_cloud(points_gt, colors_gt, title="Облако точек по эталонной диспаратности")

In [ ]:
# TODO (задание 3.2, 3.3): восстановите облако по ВАШЕЙ карте диспаратности и
# проверьте глубину по объекту с известными размерами.
#
# План:
#   1) points, colors = disparity_to_points(disp_sgbm, stereo_meta, left);
#   2) визуализируйте облако и сохраните его: save_ply(points, colors,
#      OUTPUT_DIR / "cloud_sgbm.ply");
#   3) выделите точки, попадающие в прямоугольник объекта stereo_meta["object_box"],
#      и посчитайте медианную глубину; сравните с stereo_meta["object_depth_mm"];
#   4) оцените ширину и высоту объекта по облаку (разность координат X и Y на
#      границах) и сравните с object_width_mm / object_height_mm;
#   5) повторите оценку для фона и сопоставьте относительные ошибки: по теории
#      (раздел 2.3) относительная ошибка глубины должна расти с дальностью —
#      проверьте это на своих числах;
#   6) залогируйте результат: log_run(stage="depth", ...,
#      object_depth_error_mm=..., object_width_error_mm=...).

pass

## 8. Эпиполярная геометрия и взаимное положение камер

Набор соответствий строится по неплоскому облаку трёхмерных точек, спроецированному в два вида с известными $K$, $R$, $\mathbf{t}$. В соответствия внесены гауссов шум и заданная доля выбросов — без выбросов сравнение методов оценки (8-точечный алгоритм против RANSAC/LMedS) лишено смысла.

Напоминание: для плоской сцены (например, для видов шахматной доски) оценка $F$ вырождена, и там следует использовать гомографию. Если вы работаете с собственной съёмкой, снимайте объект с выраженным рельефом.

In [ ]:
def skew(vector):
    """Кососимметричная матрица векторного произведения [v]_x."""
    x, y, z = np.asarray(vector, dtype=np.float64).ravel()
    return np.array([[0.0, -z, y], [z, 0.0, -x], [-y, x, 0.0]])


def make_two_view_correspondences(n_points=250, noise_px=0.5, outlier_ratio=0.12,
                                  image_size=(640, 480), seed=SEED):
    """Соответствия двух видов неплоской сцены с известными R, t, F.

    Выход: словарь с ключами
        'pts1', 'pts2' — (N, 2) float32 в пикселях;
        'inliers'      — (N,) bool, True для честных соответствий;
        'K', 'R_true', 't_true', 'F_true', 'E_true', 'image_size'.
    """
    generator = np.random.default_rng(seed)
    width, height = image_size
    K = np.array([[800.0, 0.0, width / 2.0],
                  [0.0, 800.0, height / 2.0],
                  [0.0, 0.0, 1.0]])
    R_true, _ = cv2.Rodrigues(np.array([0.02, -0.15, 0.01]))
    t_true = np.array([-150.0, 10.0, 30.0])

    def project(points, R, t):
        camera = points @ R.T + t
        image = camera @ K.T
        return image[:, :2] / image[:, 2:3], camera[:, 2]

    collected1, collected2 = [], []
    while len(collected1) < n_points:
        candidates = np.column_stack([
            generator.uniform(-260, 260, n_points),
            generator.uniform(-190, 190, n_points),
            generator.uniform(700, 1500, n_points),   # разброс по глубине: сцена неплоская
        ])
        p1, z1 = project(candidates, np.eye(3), np.zeros(3))
        p2, z2 = project(candidates, R_true, t_true)
        ok = ((z1 > 0) & (z2 > 0) &
              (p1[:, 0] >= 0) & (p1[:, 0] < width) & (p1[:, 1] >= 0) & (p1[:, 1] < height) &
              (p2[:, 0] >= 0) & (p2[:, 0] < width) & (p2[:, 1] >= 0) & (p2[:, 1] < height))
        collected1.extend(p1[ok])
        collected2.extend(p2[ok])
    pts1 = np.array(collected1[:n_points], dtype=np.float64)
    pts2 = np.array(collected2[:n_points], dtype=np.float64)

    pts1 += generator.normal(0.0, noise_px, pts1.shape)
    pts2 += generator.normal(0.0, noise_px, pts2.shape)

    inliers = np.ones(n_points, dtype=bool)
    n_outliers = int(round(outlier_ratio * n_points))
    if n_outliers:
        index = generator.choice(n_points, n_outliers, replace=False)
        pts2[index] = np.column_stack([generator.uniform(0, width, n_outliers),
                                       generator.uniform(0, height, n_outliers)])
        inliers[index] = False

    K_inv = np.linalg.inv(K)
    E_true = skew(t_true) @ R_true
    F_true = K_inv.T @ E_true @ K_inv
    F_true = F_true / F_true[2, 2]
    return {"pts1": pts1.astype(np.float32), "pts2": pts2.astype(np.float32),
            "inliers": inliers, "K": K, "R_true": R_true, "t_true": t_true,
            "F_true": F_true, "E_true": E_true, "image_size": image_size}


def epipolar_residual(F, pts1, pts2):
    """Невязка эпиполярного ограничения x'^T F x = 0.

    Выход: (среднее |x'^T F x|, средняя симметричная ошибка Сампсона в пикселях).
    Вторая величина имеет геометрический смысл расстояния до эпиполярной линии
    и потому сопоставима между разными нормировками F.
    """
    ones = np.ones((len(pts1), 1))
    x1 = np.hstack([np.asarray(pts1, np.float64), ones])
    x2 = np.hstack([np.asarray(pts2, np.float64), ones])
    constraint = np.einsum("ij,jk,ik->i", x2, F, x1)
    line2 = x1 @ F.T
    line1 = x2 @ F
    denominator = (line2[:, 0] ** 2 + line2[:, 1] ** 2 +
                   line1[:, 0] ** 2 + line1[:, 1] ** 2)
    sampson = constraint ** 2 / np.maximum(denominator, 1e-12)
    return float(np.mean(np.abs(constraint))), float(np.mean(np.sqrt(sampson)))


two_view = make_two_view_correspondences()
print("Соответствий:", len(two_view["pts1"]),
      "| выбросов:", int((~two_view["inliers"]).sum()))
print("Ошибка эпиполярного ограничения для истинной F (только честные пары):",
      epipolar_residual(two_view["F_true"],
                        two_view["pts1"][two_view["inliers"]],
                        two_view["pts2"][two_view["inliers"]]))

In [ ]:
def draw_epipolar(pts1, pts2, F, image_size, n_lines=12, title=""):
    """Визуализация эпиполярных линий на пустых холстах (служебная функция).

    Точки первого вида показываются слева, соответствующие им эпиполярные
    линии l' = F x — справа вместе с точками второго вида.
    """
    width, height = image_size
    canvas1 = np.full((height, width, 3), 255, np.uint8)
    canvas2 = np.full((height, width, 3), 255, np.uint8)
    selected = np.arange(min(n_lines, len(pts1)))
    lines = cv2.computeCorrespondEpilines(
        np.asarray(pts1, np.float32)[selected].reshape(-1, 1, 2), 1,
        np.asarray(F, np.float64)).reshape(-1, 3)
    colors = [(int(c[0]), int(c[1]), int(c[2]))
              for c in (np.linspace(0, 200, len(selected))[:, None] * np.array([[1, 0.4, 0.2]]))]
    for point1, point2, line, color in zip(np.asarray(pts1)[selected],
                                           np.asarray(pts2)[selected], lines, colors):
        a, b, c = line
        if abs(b) > 1e-9:
            start = (0, int(round(-c / b)))
            end = (width - 1, int(round(-(c + a * (width - 1)) / b)))
        else:
            start = (int(round(-c / a)), 0)
            end = (int(round(-c / a)), height - 1)
        cv2.line(canvas2, start, end, color, 1, cv2.LINE_AA)
        cv2.circle(canvas1, tuple(np.round(point1).astype(int)), 4, color, -1, cv2.LINE_AA)
        cv2.circle(canvas2, tuple(np.round(point2).astype(int)), 4, color, -1, cv2.LINE_AA)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].imshow(cv2.cvtColor(canvas1, cv2.COLOR_BGR2RGB))
    axes[0].set_title("вид 1: точки")
    axes[1].imshow(cv2.cvtColor(canvas2, cv2.COLOR_BGR2RGB))
    axes[1].set_title("вид 2: эпиполярные линии и точки")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


mask_in = two_view["inliers"]
draw_epipolar(two_view["pts1"][mask_in], two_view["pts2"][mask_in],
              two_view["F_true"], two_view["image_size"],
              title="Истинная фундаментальная матрица")

In [ ]:
# TODO (задание 3.4): оцените F, E и взаимное положение камер.
#
# План:
#   1) оцените F по всем соответствиям (с выбросами) тремя способами:
#      cv2.FM_8POINT, cv2.FM_RANSAC, cv2.FM_LMEDS; для каждого посчитайте
#      epipolar_residual на честных парах и долю верно определённых выбросов;
#   2) сравните оценённую F с F_true (обе нормируйте, например, по норме
#      Фробениуса или по элементу [2, 2]);
#   3) вычислите E двумя путями: E = K^T F K и cv2.findEssentialMat;
#      сравните результаты между собой и с E_true;
#   4) извлеките взаимное положение: cv2.recoverPose(E, pts1, pts2, K);
#      сравните R с R_true (угол между поворотами через cv2.Rodrigues от
#      R_est^T @ R_true) и направление t с t_true / ||t_true||;
#      объясните, почему абсолютный масштаб t восстановить нельзя;
#   5) исследуйте влияние уровня шума и доли выбросов: постройте серию
#      make_two_view_correspondences(noise_px=..., outlier_ratio=...) и
#      залогируйте ошибки для каждого метода;
#   6) визуализируйте эпиполярные линии для вашей оценки через draw_epipolar
#      и сравните картину с эталонной.
#
# Замечание: перед оценкой F точки следует нормировать (cv2.findFundamentalMat
# делает это внутри). Для собственной реализации 8-точечного алгоритма
# нормировка Хартли обязательна, иначе решение будет плохо обусловлено.

runs_table()

## Отчёт

### Сводные таблицы

Обязательны:

1. калибровка: число видов → RMS репроекции, отклонение $f_x, f_y, c_x, c_y$ от истины (для реальной съёмки — только RMS и распределение по видам);
2. сопоставление: конфигурация → плотность, MAE, RMSE, доля грубых ошибок, время;
3. глубина: объект → истинная глубина и размер, восстановленные значения, абсолютная и относительная ошибка;
4. эпиполярная геометрия: метод оценки → ошибка эпиполярного ограничения, ошибка $R$ в градусах, угол между направлениями $\mathbf{t}$.

In [ ]:
frame = runs_table()

if frame.empty:
    print("Журнал пуст: выполните разделы 5–8.")
else:
    for stage, group in frame.groupby("stage"):
        print(f"\n=== {stage} ===")
        columns = [c for c in group.columns
                   if c not in {"run_id", "stage"} and group[c].notna().any()]
        display(group[columns].round(4))

    disparity_runs = frame[frame.get("stage") == "disparity"] if "stage" in frame else pd.DataFrame()
    if not disparity_runs.empty and {"density", "bad_share"} <= set(disparity_runs.columns):
        fig, ax = plt.subplots(figsize=(6.5, 4.5))
        ax.scatter(disparity_runs["density"], disparity_runs["bad_share"])
        for row in disparity_runs.itertuples():
            ax.annotate(str(row.params), (row.density, row.bad_share),
                        textcoords="offset points", xytext=(4, 4), fontsize=7)
        ax.set_xlabel("плотность карты")
        ax.set_ylabel("доля грубых ошибок")
        ax.set_title("компромисс плотность/качество карты диспаратности")
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

save_runs()

### Выводы

**Наблюдения** (измеренные факты со ссылкой на строки журнала: ошибка репроекции, метрики карт, ошибки глубины и позы):

-

**Интерпретация** (чем объясняются ошибки: зоны перекрытия, однородные области, размер блока, ограниченный диапазон диспаратности, шум и выбросы в соответствиях):

-

**Выводы и их границы** (что проверено на синтетике и что из этого не переносится на реальную съёмку: дисторсия, неточность ректификации, рассинхронизация камер, неизвестная база; на каких дальностях оценка глубины ещё осмысленна при вашей базе и фокусном расстоянии):

-

**Ограничения стереометода** — обязательный пункт задания. Опирайтесь на измерения: рост погрешности глубины с дальностью, доля неопределённых пикселей в зонах перекрытия и на однородных участках, минимальная восстанавливаемая дальность при выбранном `numDisparities`.

## Контрольные вопросы

Из [списка вопросов блока](README.md#контрольные-вопросы-блока):

8. Что такое эпиполярная линия и как она используется при поиске соответствий?
9. Чем фундаментальная матрица отличается от существенной?
10. Как из карты диспаратности получить глубину и какие параметры камеры для этого нужны?

Дополнительно:

- Как изменится карта диспаратности при уменьшении блока сопоставления (вопрос к защите)?
- Почему по двум видам нельзя определить абсолютный масштаб перемещения камеры?
- Почему оценка фундаментальной матрицы вырождена для плоской сцены?

## Чек-лист перед сдачей

- [ ] Ноутбук исполняется сверху вниз без ошибок после `Restart & Run All`.
- [ ] Указаны ФИО, группа, номер работы, источники данных (синтетика и/или собственная съёмка).
- [ ] Seed и версии библиотек зафиксированы и выведены.
- [ ] Калибровка выполнена; указана RMS ошибки репроекции и её распределение по видам.
- [ ] Полученная матрица внутренних параметров сопоставлена с эталоном (для синтетики) или проверена на согласованность (для реальной съёмки).
- [ ] Карта диспаратности построена; учтено масштабирование результата `compute()` на 16 и отрицательные значения для ненайденных пикселей.
- [ ] Проведена серия по параметрам сопоставления; для каждой конфигурации приведены плотность и качество.
- [ ] Сравнены StereoBM и StereoSGBM при сопоставимых параметрах.
- [ ] Восстановлено и визуализировано облако точек, файл PLY сохранён.
- [ ] Глубина проверена по объекту с известными размерами; приведены абсолютная и относительная ошибки.
- [ ] Оценены F и E, извлечено взаимное положение камер, ошибки сопоставлены с истиной.
- [ ] Есть сводные таблицы; журнал сохранён (`outputs_lab4/runs.csv`).
- [ ] Наблюдения, интерпретация и выводы разделены; сформулированы ограничения стереометода.